In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE, get_generate_prompt

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/check_confirmation_bias'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def mix_contexts(contexts_fact_dict: dict, contexts_counter_dict: dict, ext_n_fact: int, ext_n_counter: int):
    if ext_n_fact <= len(contexts_fact_dict) and ext_n_counter <= len(contexts_counter_dict):
        ext_contexts_fact = random.sample(list(contexts_fact_dict.values()), ext_n_fact)
        ext_contexts_counter = random.sample(list(contexts_counter_dict.values()), ext_n_counter)

        # 셔플 전 각각의 컨텍스트에 태그(출처)를 붙여 튜플 형태로 결합
        tagged_contexts = [(ctx, 'fact') for ctx in ext_contexts_fact] + [(ctx, 'counter') for ctx in ext_contexts_counter]

        # 태그를 붙인 상태에서 셔플
        random.shuffle(tagged_contexts)

        # 태그를 제거하고 위치 기록
        mixed_contexts = []
        fact_idxs, counter_idxs = [], []

        for i, (ctx, tag) in enumerate(tagged_contexts):
            mixed_contexts.append(ctx)
            if tag == 'fact':
                fact_idxs.append(i)
            else:
                counter_idxs.append(i)

        return mixed_contexts, fact_idxs, counter_idxs

    return None, None, None

In [ ]:
# contexts_fact_dict = {
#     '1': 'fact context 1',
#     '2': 'fact context 2',
#     '3': 'fact context 3',
#     '4': 'fact context 4'
# }

# contexts_counter_dict = {
#     '4': 'counter context 4',
#     '5': 'counter context 5',
#     '6': 'counter context 6',
#     '7': 'counter context 7'
# }

# mixed_contexts, fact_idxs, counter_idxs = mix_contexts(contexts_fact_dict, contexts_counter_dict, 2, 3)

# print(f'mixed_contexts : {mixed_contexts}\n')
# print(f'fact_idxs : {fact_idxs}\n')
# print(f'counter_idxs : {counter_idxs}')